In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
import os
import wandb

# Force W&B to stay on your machine only
os.environ["WANDB_MODE"] = "offline" 

wandb.init(project="cifar10-local")




In [2]:

# Initialize W&B
wandb.init(project="cifar10-2026-boost", config={
    "learning_rate": 0.001,
    "batch_size": 512,
    "epochs": 10,
})

# Modern data augmentation for CIFAR-10
transform = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])

train_set = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_set, batch_size=512, shuffle=True, num_workers=8)

c:\Users\vinay\OneDrive\Desktop\VATSA\VATSA\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


In [3]:
from torchvision.models import efficientnet_b0

device = "cuda"
model = efficientnet_b0(num_classes=10).to(device)

# The "2026 speed boost" line
# Note: First iteration will be slow due to compilation
model = torch.compile(model) 

criterion = torch.nn.CrossEntropyLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3)
scaler = torch.cuda.amp.GradScaler() # GradScaler for Mixed Precision


C:\Users\vinay\AppData\Local\Temp\ipykernel_40388\3861484208.py:12: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler() # GradScaler for Mixed Precision


In [4]:
for epoch in range(50):
    model.train()
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()

        # Autocast for Mixed Precision
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            outputs = model(images)
            loss = criterion(outputs, labels)

        # Scaled backward pass
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

    # Log metrics to W&B dashboard
    wandb.log({"epoch": epoch, "loss": loss.item()})


In [5]:
# Save model locally
torch.save(model.state_dict(), "best_model.pth")

# Log model to W&B as an artifact
artifact = wandb.Artifact('cifar10-model', type='model')
artifact.add_file('best_model.pth')
wandb.log_artifact(artifact)



<Artifact cifar10-model>

In [6]:

embeddings_list = []
labels_list = []

model.eval()
print("Generating embeddings on T4...")


Generating embeddings on T4...


In [7]:

import torch.nn as nn
with torch.no_grad():
    # Use AMP for speed on T4
    with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
        # Taking 5 batches for the embedding file
        for i, (images, labels) in enumerate(train_loader):
             
            
            # Move data to GPU
            images = images.to(device)
            
            # Extract features (penultimate layer)
            # For EfficientNet, .features gives the convolutional base
            feats = model.features(images)
            feats = nn.functional.adaptive_avg_pool2d(feats, (1, 1)).flatten(1)
            
            embeddings_list.append(feats.cpu())
            labels_list.append(labels)
            print(f"Batch {i} processed", end="\r")

# 2. Concatenate and Save
final_embeddings = torch.cat(embeddings_list)
final_labels = torch.cat(labels_list)

torch.save({
    'embeddings': final_embeddings,
    'labels': final_labels
}, "cifar10_embeddings.pt")

torch.save(model.state_dict(), "cifar10_best_model.pth")

print(f"\nSuccess! Saved {final_embeddings.shape[0]} embeddings to cifar10_embeddings.pt")

Batch 97 processed
Success! Saved 50000 embeddings to cifar10_embeddings.pt


In [1]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.models import efficientnet_b0

# 1. Setup Test Data (Use same transforms as training)
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5))
])
test_set = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transform)
test_loader = torch.utils.data.DataLoader(test_set, batch_size=128, shuffle=False)

# 2. Reload Model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = efficientnet_b0(num_classes=10).to(device)

# 1. Load the state dict
state_dict = torch.load("cifar10_best_model.pth")

# 2. Create a new state dict without the '_orig_mod.' prefix
from collections import OrderedDict
new_state_dict = OrderedDict()
for k, v in state_dict.items():
    name = k.replace("_orig_mod.", "") # remove the prefix added by torch.compile
    new_state_dict[name] = v


# 3. Load it into your model
model.load_state_dict(new_state_dict)
model.eval()
print("Model loaded successfully!")

# 3. Accuracy Loop
correct = 0
total = 0

with torch.no_grad(): # Speeds up inference
    for images, labels in test_loader:
        images, labels = images.to(device), labels.to(device)
        outputs = model(images)
        _, predicted = torch.max(outputs.data, 1) # Get highest probability class
        total += labels.size(0)
        correct += (predicted == labels).sum().item()

accuracy = 100 * correct / total
print(f'Accuracy of the model on the 10,000 test images: {accuracy:.2f}%')


c:\Users\vinay\OneDrive\Desktop\VATSA\VATSA\.venv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


Model loaded successfully!
Accuracy of the model on the 10,000 test images: 67.74%


In [6]:
import numpy as np
from tqdm import tqdm

print("\nExtracting embeddings...")

all_embeddings = []
all_labels = []
all_preds = []

with torch.no_grad():
    for images, labels in tqdm(test_loader):
        images = images.to(device)

        logits, embeddings = model(images)

        preds = logits.argmax(1).cpu().numpy()

        all_embeddings.append(embeddings.cpu().numpy())
        all_labels.append(labels.numpy())
        all_preds.append(preds)

all_embeddings = np.vstack(all_embeddings)
all_labels = np.hstack(all_labels)
all_preds = np.hstack(all_preds)

print("Embeddings shape:", all_embeddings.shape)


Extracting embeddings...


  0%|          | 0/79 [00:00<?, ?it/s]


ValueError: too many values to unpack (expected 2)